# Stacking

-  Train multiple models
-  Use the predictions of these models as features for a meta-model
-  Train the meta-model on these features

# Naive Bayes

- Let's say we are seperating spam and non spam emails
    - P(Spam | sentence) = P(Spam) * P(sentence | Spam) / P(sentence)
    - P(Non Spam | sentence) = P(Non Spam) * P(sentence | Non Spam) / P(sentence)
    - If P(Spam | sentence) > P(Non Spam | sentence), then the email is spam and vice versa

- sentence -> word1 + word2 + ... wordN (here all the stop words are removed and words are lemmatized)
- Naive bayes makes the **Naive** assumption that the words are independent of each other
- and calculates P(sentence | Spam) = P(word1 | Spam) * P(word2 | Spam) * ... * P(wordN | Spam) 


- Laplace Smoothing
    - If a new word occurs, instead of making it 0, we calculate the probability as
    P(wj/spam) = (count(wj/spam) + alpha) / (count(spam) + alpha * V), where V is the number of labels(in our case 2 since it is either spam or not spam(ham)) and alpha is a small constant, it is a hyperparameter.
    - Instead of doing this for only unkown words/new words, we do this for every word
    - Since we are multiplying probabilities, there is a good chance , values get very close to 0, causing the underflow problem(It's tough for computers to compare small floating point numbers)
    - To avoid this, we take the log of the probabilities and then add them

- Feature Importance --> Higher the value of P(wj/spam(or ham)), higher the importance of the word

In [2]:
x = 2/504
print(x)

0.003968253968253968


In [3]:
data_list = [("This process, however, afforded me no means of...", "EAP"), ("It never once occurred to me that the fumbling...", "HPL"), ("In his left hand was a gold snuff box, from wh...", "EAP"),("How lovely is spring As we looked from Windsor...", "MWS"),("Finding nothing else, not even gold, the Super...", "HPL"),("A youth passed in solitude, my best years spen...", "MWS"),("The astronomer, perhaps, at this point, took r...", "EAP"),("The surcingle hung in ribands from my body.", "EAP"),("I knew that you could not say to yourself 'ste...", "EAP"),("I confess that neither the structure of langua...", "MWS")]

In [6]:
import pandas as pd
import numpy as np
data_list = [("This process, however, afforded me no means of...", "EAP"), ("It never once occurred to me that the fumbling...", "HPL"), ("In his left hand was a gold snuff box, from wh...", "EAP"),("How lovely is spring As we looked from Windsor...", "MWS"),("Finding nothing else, not even gold, the Super...", "HPL"),("A youth passed in solitude, my best years spen...", "MWS"),("The astronomer, perhaps, at this point, took r...", "EAP"),("The surcingle hung in ribands from my body.", "EAP"),("I knew that you could not say to yourself 'ste...", "EAP"),("I confess that neither the structure of langua...", "MWS")]
data_list = np.array(data_list)
eap = len(data_list[data_list[:,1] == 'EAP'])
hpl = len(data_list[data_list[:,1] == 'HPL'])
mws = len(data_list[data_list[:,1] == 'MWS'])

p_eap = eap / len(data_list)
p_hpl = hpl / len(data_list)
p_mws = mws / len(data_list)
print(p_eap)
print(p_hpl)
print(p_mws)

0.5
0.2
0.3


In [13]:
import numpy as np
import pandas as pd

def class_priors(df):

  # Finding number of EAP     
  eap = len(df[df['label'] == 'EAP'])
  
  # Finding number of HPL
  hpl = len(df[df['label'] == 'HPL'])
  
  # Finding number of MWS
  mws = len(df[df['label'] == 'MWS'])
  
  # Total number of entries
  total_entries = len(df)

  # Class Prior for EAP    
  class_prior_eap = eap/len(df)
  
  # Class Prior for HPL
  class_prior_hpl = hpl/len(df)
  
  # Class Prior for MWS
  class_prior_mws = mws/len(df)

  return (class_prior_eap, class_prior_hpl, class_prior_mws)

data_list = [("This process, however, afforded me no means of...", "EAP"), ("It never once occurred to me that the fumbling...", "HPL"), ("In his left hand was a gold snuff box, from wh...", "EAP"),("How lovely is spring As we looked from Windsor...", "MWS"),("Finding nothing else, not even gold, the Super...", "HPL"),("A youth passed in solitude, my best years spen...", "MWS"),("The astronomer, perhaps, at this point, took r...", "EAP"),("The surcingle hung in ribands from my body.", "EAP"),("I knew that you could not say to yourself 'ste...", "EAP"),("I confess that neither the structure of langua...", "MWS")]

df = pd.DataFrame(data_list, columns=['text', 'label'])
print(df)
print(class_priors(df))

                                                text label
0  This process, however, afforded me no means of...   EAP
1  It never once occurred to me that the fumbling...   HPL
2  In his left hand was a gold snuff box, from wh...   EAP
3  How lovely is spring As we looked from Windsor...   MWS
4  Finding nothing else, not even gold, the Super...   HPL
5  A youth passed in solitude, my best years spen...   MWS
6  The astronomer, perhaps, at this point, took r...   EAP
7        The surcingle hung in ribands from my body.   EAP
8  I knew that you could not say to yourself 'ste...   EAP
9  I confess that neither the structure of langua...   MWS
(0.5, 0.2, 0.3)


In [14]:
X_train = [[1, 0, 1, 2, 0], [3, 0, 1, 3, 0], [0, 2, 1, 1, 3], [1, 2, 0, 0, 1], [0, 1, 1, 3, 0], [2, 0, 2, 3, 1], [3, 1, 2, 1, 3], [0, 0, 0, 1, 1], [0, 2, 1, 1, 3], [2, 2, 3, 1, 1], [3, 2, 3, 1, 1], [0, 2, 1, 2, 0], [0, 0, 0, 0, 2], [1, 0, 0, 0, 2], [3, 2, 1, 2, 2], [2, 3, 0, 0, 1], [2, 3, 3, 1, 3], [0, 1, 2, 0, 1], [1, 2, 0, 1, 3], [1, 3, 1, 2, 0], [1, 3, 2, 2, 0], [3, 0, 0, 1, 2], [3, 2, 1, 2, 0], [1, 3, 1, 2, 0], [3, 3, 2, 3, 1], [3, 0, 2, 1, 3], [0, 3, 1, 0, 0], [1, 0, 0, 0, 3], [0, 2, 0, 1, 0], [2, 1, 3, 1, 3], [3, 2, 1, 1, 1], [1, 0, 1, 1, 1], [3, 0, 0, 2, 3], [2, 2, 2, 1, 0], [3, 3, 2, 2, 1], [1, 2, 2, 3, 0], [0, 3, 1, 2, 0], [3, 1, 1, 3, 3], [1, 3, 2, 2, 0], [2, 3, 3, 3, 0], [2, 3, 2, 1, 3], [2, 0, 1, 2, 1], [0, 1, 2, 3, 2], [3, 1, 2, 0, 2], [2, 2, 3, 0, 1], [2, 1, 0, 3, 0], [0, 1, 1, 0, 0], [2, 3, 2, 2, 2], [3, 2, 1, 2, 3], [3, 2, 0, 0, 0]]
y_train = [0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0]

X_test = [[0, 2, 1, 1, 1], [2, 1, 1, 0, 0], [2, 0, 2, 2, 2], [2, 0, 3, 1, 0], [3, 3, 1, 2, 1], [3, 0, 1, 0, 2], [0, 3, 3, 3, 1], [3, 3, 0, 3, 2], [0, 3, 2, 2, 3], [2, 3, 3, 0, 0]]
y_test = [0, 1, 1, 0, 0, 1, 0, 1, 0, 0]

alpha = [0.1, 1, 100, 1000]

In [15]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

def train_and_predict(alpha):
    nb_classifier = MultinomialNB(alpha=alpha)
    nb_classifier.fit(X_train, y_train)
    y_pred = nb_classifier.predict(X_test)
    return accuracy_score(y_test, y_pred)


for alpha in alpha:
    print("Alpha: ", alpha)
    print("Accuracy: ", train_and_predict(alpha))
    print()


Alpha:  0.1
Accuracy:  0.9

Alpha:  1
Accuracy:  0.9

Alpha:  100
Accuracy:  0.9

Alpha:  1000
Accuracy:  0.6



In [16]:
x = 0.9*0.6/(0.9*0.6+0.1*0.4)
print(x)

0.9310344827586207


In [17]:
import numpy as np

def solve(prior,positive_covid,positive_not_covid):
    # YOUR CODE GOES HERE
    ans= positive_covid*prior/(positive_covid*prior+positive_not_covid*(1-prior))
    return ans;

print(solve(0.6,0.9,0.1))


0.9310344827586207


In [21]:
x = 2/3 * 3/4 * 1/4 * 3/4
y = 10.9/101.8
print(y)
print(x)

0.10707269155206288
0.09375
